Задача

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:

df = pd.read_csv("data/task_2_data_ex.csv")

In [6]:
fin  = df[df['produced_material_release_type'] == 'FIN']
prod = df[df['produced_material_release_type'] != 'FIN']


In [7]:
# 1. Start with your FIN materials
current_level = fin.copy()
all_layers = []

while not current_level.empty:
    # Add the current batch of rows to our collection
    all_layers.append(current_level)
    
    # 2. Find the "Children": Search for the next step in the 'prod' dataframe
    # We match the 'component_material' of the current step 
    # to the 'produced_material' of the next step
    current_level = prod.merge(
        current_level[['plant_id', 'year', 'month', 'component_material']].drop_duplicates(),
        left_on=['plant_id', 'year', 'month', 'produced_material'],
        right_on=['plant_id', 'year', 'month', 'component_material']
    )
    
    # Clean up the merge columns so it's ready for the next loop
    current_level = current_level.drop(columns=['component_material_y']).rename(
        columns={'component_material_x': 'component_material'}
    )

# 3. Stack everything together
final_output = pd.concat(all_layers, ignore_index=True)

In [8]:
# Select only the columns needed for the report
final_report = final_output[[
    'plant_id', 'year', 'produced_material', 
    'produced_material_release_type', 'produced_material_production_type', 
    'component_material'
]]

# Rename to the clean headers you specified
final_report.columns = ['Plant', 'Year', 'Material', 'Release_Type', 'Prod_Type', 'Component']

In [9]:
print(final_report)

       Plant  Year  Material Release_Type  Prod_Type  Component
0     RLT_10  2024     10000          FIN       8002      50000
1     RLT_10  2024     10000          FIN       8002      50000
2     RLT_10  2024     10000          FIN       8002      50000
3     RLT_10  2024     10000          FIN       8002      50000
4     RLT_10  2024     10000          FIN       8002      50000
...      ...   ...       ...          ...        ...        ...
1315  RLT_14  2024     80009         PROD       8000      90050
1316  RLT_14  2024     80009         PROD       8000      70009
1317  RLT_14  2024     80009         PROD       8000      90050
1318  RLT_14  2024     80009         PROD       8000      70009
1319  RLT_14  2024     80009         PROD       8000      90050

[1320 rows x 6 columns]


In [16]:
# Use double quotes for the f-string, and avoid ":" in filenames
df.to_csv(f"final_result_{datetime.today().strftime('%Y-%m-%d_%H-%M-%S')}.csv", index=False)